

# Lab 3 · Scraper Detective
### Does your pipeline see what *you* see?

*Solo lab · about 2 hours · Colab · your own Google Cloud bucket*

Your Craigslist robot has been running on Google Cloud. Tonight you play **detective**: you pull your own
data into this notebook, **search** it, open the **real ads** side by side, **grade** what the regex extractor got right
and wrong, and then **fix the worst field with one regular expression.**

| Part | Time | What you do |
|---|---|---|
| **1. Get it running** | 30 min | Finish the Final GCP Guide F26 if you have not yet: 5 green checks, then one scrape |
| **2. Search and see** | 25 min | Load your bucket here, search your cars, open 3 live ads |
| **3. Grade the robot** | 30 min | A truth table for 5 cars: what the regex extracted vs what the ad really says |
| **4. Fix it with regex** | 30 min | Write ONE better regular expression for make and model, and re-grade |
| **5. Wrap up** | 5 min | Three short answers, then save your notebook |

**Stuck?** Ask Claude. Paste the cell, the full error, and what you expected to happen.

# Part 1 · Get it running (30 min)

Skip to Part 2 if you already have **5 green checks** and at least one scrape in your bucket.

1. Follow the **[Final GCP Guide F26](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module3/FINAL_GCP_Guide_F26.ipynb)** (it is on HuskyCT, Week 3.1) through Step 10.
2. In **Cloud Scheduler**, Force run these **one at a time**, waiting for **Success** each time
   (tick the checkbox first, then **Force run**):
   1. `craigslist-scraper-hourly`
   2. `extractor-per-listing-hourly`
   3. `materialize-master-hourly`
3. In **Cloud Storage**, open your bucket: you should see `scrapes/` (raw ads) and `structured/` (the extracted fields).

Tip: run the scraper once or twice more before class (an hour apart), so you have 20 to 30 cars to play with.

# Part 2 · Search and see (25 min)

## 2a. Connect this notebook to your bucket
When the pop-up asks which Google account to use, pick the **same Gmail you used for Google Cloud**
(not necessarily your UConn account). Then type your two values below. **These are the only two lines you edit in Part 2.**

In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "YOUR-PROJECT-ID"      # e.g. "myscrapers-dww05002-f26-510015"
BUCKET_NAME = "YOUR-PROJECT-ID"     # usually the same as your project ID

In [ ]:
import io
import re
import pandas as pd
from google.cloud import storage

client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)

n_txt = 0
n_jsonl = 0
for blob in client.list_blobs(BUCKET_NAME):
    if blob.name.startswith("scrapes/") and blob.name.endswith(".txt"):
        n_txt = n_txt + 1
    if blob.name.startswith("structured/") and blob.name.endswith(".jsonl"):
        n_jsonl = n_jsonl + 1
print("Raw ads saved (scrapes/*.txt):       ", n_txt)
print("Extracted records (structured/*.jsonl):", n_jsonl)

## 2b. Load your cars
`listings_master.csv` is the tidy table your pipeline builds every hour. We also attach each car's **Craigslist link**
(from the scraper's `index.csv` files) so you can open the real ad.

In [ ]:
master_blob = bucket.blob("structured/datasets/listings_master.csv")
cars = pd.read_csv(io.BytesIO(master_blob.download_as_bytes()), dtype={"post_id": str})

link_tables = []
for blob in client.list_blobs(BUCKET_NAME, prefix="scrapes/"):
    if blob.name.endswith("index.csv"):
        one = pd.read_csv(io.BytesIO(blob.download_as_bytes()), dtype={"post_id": str})
        link_tables.append(one)
links = pd.concat(link_tables, ignore_index=True)
links = links[["post_id", "url"]].drop_duplicates("post_id")

cars = cars.merge(links, on="post_id", how="left")
print(len(cars), "cars")
cars[["post_id", "price", "year", "make", "model", "mileage", "url"]].head(10)

**Look at the `make` and `model` columns.** Anything strange? Keep that thought: it is the mystery for Part 3.

## 2c. Search your cars
The `make` column is not trustworthy yet, so we search the **raw ad text** instead. Change `SEARCH_WORD` and re-run.
Try a brand (`toyota`, `honda`, `ford`), a feature (`sunroof`, `4wd`), or a word like `clean title`.

In [ ]:
def read_ad_text(source_txt):
    return bucket.blob(source_txt).download_as_text()


SEARCH_WORD = "toyota"

matches = []
for i in range(len(cars)):
    row = cars.iloc[i]
    text = read_ad_text(row["source_txt"])
    if SEARCH_WORD.lower() in text.lower():
        matches.append(row)

print(len(matches), "of", len(cars), "ads mention", repr(SEARCH_WORD))
for row in matches:
    print(row["price"], row["year"], row["mileage"], row["url"])

## 2d. See it with your own eyes
Pick **3 cars**. For each one, open its `url` in a new tab **and** print its raw text below.
Scroll the raw text: can you find the price, the year, the make and model, and the odometer?

In [ ]:
POST_ID = cars.iloc[0]["post_id"]     # change to any post_id from your table

row = cars[cars["post_id"] == POST_ID].iloc[0]
print(row["url"])
print("=" * 80)
print(read_ad_text(row["source_txt"])[:3000])

**Write 2 or 3 sentences:** what does the raw text keep from the web page, and what does it lose? (photos, layout, ...)

*Your answer:*

# Part 3 · Grade the robot (30 min)

The extractor (Module 3.2) uses three regular expressions and some mileage rules to turn each raw ad into
`price, year, make, model, mileage`. How good is it? You are going to find out with a **truth table**.

## 3a. Pick 5 cars and write down the truth
The cell below prints 5 of your cars with **what the robot extracted** and the **link**. Open each ad and read the real
values. Craigslist lists them near the photos (for example `2017` on one line, then `chevrolet colorado`, and `odometer: 188,500`).

In [ ]:
graded = cars.head(5)

for i in range(len(graded)):
    row = graded.iloc[i]
    print(row["post_id"], "| robot says:", row["price"], row["year"], row["make"], row["model"], row["mileage"])
    print("   ", row["url"])

print()
print("Copy this into the next cell and fill in the TRUE values from each ad:")
print("truth = {")
for i in range(len(graded)):
    pid = graded.iloc[i]["post_id"]
    print('    "' + pid + '": {"price": None, "year": None, "make": "", "model": "", "mileage": None},')
print("}")

In [ ]:
# Paste the dictionary from above and type the TRUE values from each ad.
# Numbers without $ or commas (4450, not "$4,450"). Make and model in lowercase, first word only for model
# (for "chevrolet colorado lt" use make "chevrolet" and model "colorado").
truth = {
}

## 3b. Score it

In [ ]:
FIELDS = ["price", "year", "make", "model", "mileage"]


def same(robot_value, true_value, field):
    if field in ["make", "model"]:
        return str(robot_value).strip().lower() == str(true_value).strip().lower()
    try:
        return float(robot_value) == float(true_value)
    except (TypeError, ValueError):
        return False


def score(robot_table, truth):
    rows = []
    for field in FIELDS:
        right = 0
        for pid in truth:
            robot_row = robot_table[robot_table["post_id"] == pid].iloc[0]
            if same(robot_row[field], truth[pid][field], field):
                right = right + 1
        rows.append({"field": field, "correct": right, "out_of": len(truth), "accuracy": right / max(len(truth), 1)})
    return pd.DataFrame(rows)


before = score(cars, truth)
before

**Which field is the most reliable? Which is the worst?**

## 3c. Why does make/model fail?
Here is the extractor's make/model rule, straight from `cloud_function/extractor-per-listing/main.py`:

```python
MAKE_MODEL_RE = re.compile(r"\b([A-Z][a-z]+)\s+([A-Z][A-Za-z0-9]+)")
```

In words: *the first place where one Capitalized word is followed by another Capitalized word.*
Run the cell to see what it grabs from one of your ads, and the lines where the real answer lives.

In [ ]:
MAKE_MODEL_RE = re.compile(r"\b([A-Z][a-z]+)\s+([A-Z][A-Za-z0-9]+)")

text = read_ad_text(graded.iloc[0]["source_txt"])
found = MAKE_MODEL_RE.search(text)
print("The robot's regex grabs:", found.groups() if found else None)
print()

ad_lines = text.split("\n")
for i in range(len(ad_lines)):
    if "odometer" in ad_lines[i].lower():
        start = max(0, i - 14)
        for j in range(start, i + 2):
            print(repr(ad_lines[j]))
        break

Look at the printed lines (the `repr` shows each line exactly, including where it ends).
Near the top of that block there is a line that is **just a year**, and the very next line is the **make and model**
in lowercase. That pattern is your clue.

# Part 4 · Fix it with one regular expression (30 min)

Write **one** regular expression that finds:
- a line that is **only a 4-digit year** (starting with 19 or 20),
- then, on the **next line**, the **make** (first word) and the **model** (second word).

Put the make in capture group 1 and the model in capture group 2. Helpful pieces:

| Piece | Means |
|---|---|
| `^` and `$` with `re.MULTILINE` | start / end of a **line** |
| `(?:19\|20)\d{2}` | a year like 1998 or 2017 (the `(?: )` groups without capturing) |
| `\n` | the line break between the two lines |
| `([A-Za-z-]+)` | one word of letters (and hyphens, for `mercedes-benz`) |
| `\s+` | one or more spaces |

Test ideas fast at **regex101.com** (pick the Python flavor), or ask Claude to explain a pattern piece by piece.
Your instructor will build one live with you in class, too.

In [ ]:
BETTER_MAKE_MODEL_RE = re.compile(r"YOUR PATTERN HERE", re.MULTILINE)   # <-- your one line


def better_make_model(text):
    found = BETTER_MAKE_MODEL_RE.search(text)
    if found is None:
        return "", ""
    return found.group(1).lower(), found.group(2).lower()


print(better_make_model(read_ad_text(graded.iloc[0]["source_txt"])))

## 4b. Re-grade with your regex
We re-run your regex on the same 5 raw ads and score the table again. Did make and model go up?

In [ ]:
fixed = graded.copy()
for i in range(len(fixed)):
    text = read_ad_text(fixed.iloc[i]["source_txt"])
    make, model = better_make_model(text)
    fixed.iloc[i, fixed.columns.get_loc("make")] = make
    fixed.iloc[i, fixed.columns.get_loc("model")] = model

after = score(fixed, truth)
comparison = before[["field", "accuracy"]].rename(columns={"accuracy": "before"})
comparison["after"] = after["accuracy"]
comparison

**Stretch (if you have time):** run your regex on **all** your cars (not just the 5) and count how many it
could not match at all. Open one of those ads: why did it miss?

In [ ]:
missed = []
for i in range(len(cars)):
    text = read_ad_text(cars.iloc[i]["source_txt"])
    make, model = better_make_model(text)
    if make == "":
        missed.append(cars.iloc[i]["url"])
print(len(missed), "of", len(cars), "ads had no match")
for url in missed:
    print(url)

# Part 5 · Wrap up (5 min)

Answer in 1 or 2 sentences each:

1. Which field did the original robot get **most** right, and which **least**? Why?
2. Where could **your** regex still break? (Think about how different people post their ads.)
3. In Module 3.3 you will let an LLM (Gemini on Vertex) read the ad instead of a regex. What will it do **better**, and what could go **wrong**?

*Your answers:*

---

**Save your work:** File → Download → `.ipynb`, and submit it as instructed on HuskyCT.
Keep your pipeline running: the more hours it scrapes, the better your data for the midterm project.